# Choquet time-series workflow with `sktime`

This notebook follows a compact forecasting workflow: inspect the series, decide on transformations and trend removal, inspect dependence, select the model with temporal cross-validation, and evaluate a genuine hold-out period.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.compose import TransformedTargetForecaster
from sktime.forecasting.auto_reg import AutoREG
from sktime.forecasting.model_selection import ForecastingGridSearchCV
from sktime.forecasting.trend import PolynomialTrendForecaster
from sktime.performance_metrics.forecasting import (
    MeanAbsoluteError,
    MeanAbsolutePercentageError,
)
from sktime.split import ExpandingWindowSplitter, temporal_train_test_split
from sktime.transformations.series.boxcox import LogTransformer
from sktime.transformations.series.detrend import Detrender

from capacities_ml_fin.base.interpretation import (
    pairwise_interactions,
    shapley_indices,
)
from capacities_ml_fin.ml.models import ChoquetAutoRegressor

## 1. Data and temporal hold-out

The example is positive and has multiplicative growth, so a logarithm is a reasonable candidate. The test set is separated before model selection to prevent future information from entering the analysis.

In [ ]:
rng = np.random.default_rng(7)
index = pd.period_range("2015-01", periods=120, freq="M")
log_values = np.empty(len(index))
log_values[0] = 3.0
for time in range(1, len(index)):
    deviation = log_values[time - 1] - (3.0 + 0.006 * (time - 1))
    log_values[time] = 3.0 + 0.006 * time + 0.65 * deviation + rng.normal(0, 0.035)

y = pd.Series(np.exp(log_values), index=index, name="value")
y_train, y_test = temporal_train_test_split(y, test_size=12)
print(f"Training observations: {len(y_train)}")
print(f"Test observations: {len(y_test)}")
y_train.plot(title="Training series", figsize=(10, 3));

## 2. Transformation, trend, ACF and PACF

Compare levels and logs rather than applying a log automatically. Here the log stabilizes the scale. We remove a simple linear trend only for the diagnostic plots; the final `sktime` pipeline will learn and reverse these transformations without leaking the test set.

In [ ]:
log_train = np.log(y_train)
time = np.arange(len(log_train))
trend_coefficients = np.polyfit(time, log_train.to_numpy(), deg=1)
detrended_log = pd.Series(
    log_train.to_numpy() - np.polyval(trend_coefficients, time),
    index=log_train.index,
    name="detrended_log_value",
)

figure, axes = plt.subplots(1, 2, figsize=(11, 3))
y_train.plot(ax=axes[0], title="Level")
log_train.plot(ax=axes[1], title="Log level")
plt.tight_layout()

adf_statistic, adf_pvalue, *_ = adfuller(detrended_log)
print(f"ADF statistic after log and detrending: {adf_statistic:.3f}")
print(f"ADF p-value: {adf_pvalue:.4f}")

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3))
plot_acf(detrended_log, lags=18, ax=axes[0])
plot_pacf(detrended_log, lags=18, method="ywm", ax=axes[1])
plt.tight_layout()

## 3. A reversible `sktime` pipeline

`TransformedTargetForecaster` applies the log and detrending to `y`, fits a forecaster on the transformed series, and reverses both operations when forecasting. The Choquet model and the classical AR model use the same transformations, validation windows, scoring rule, and candidate lag orders. `AutoREG` is the `sktime` wrapper around the classical `statsmodels` autoregression.

In [ ]:
choquet_pipeline = TransformedTargetForecaster(
    steps=[
        ("log", LogTransformer()),
        ("detrend", Detrender(forecaster=PolynomialTrendForecaster(degree=1))),
        ("forecast", ChoquetAutoRegressor()),
    ]
)
classical_ar_pipeline = TransformedTargetForecaster(
    steps=[
        ("log", LogTransformer()),
        ("detrend", Detrender(forecaster=PolynomialTrendForecaster(degree=1))),
        ("forecast", AutoREG(lags=1, trend="c")),
    ]
)

validation = ExpandingWindowSplitter(
    initial_window=60, step_length=12, fh=[1, 2, 3]
)
search = ForecastingGridSearchCV(
    forecaster=choquet_pipeline,
    cv=validation,
    param_grid={"forecast__lags": [1, 2, 3]},
    scoring=MeanAbsoluteError(),
)
search.fit(y_train)
classical_search = ForecastingGridSearchCV(
    forecaster=classical_ar_pipeline,
    cv=validation,
    param_grid={"forecast__lags": [1, 2, 3]},
    scoring=MeanAbsoluteError(),
)
classical_search.fit(y_train)

print("Best Choquet parameters:", search.best_params_)
print("Best classical AR parameters:", classical_search.best_params_)
display(
    search.cv_results_[
        ["params", "mean_test_MeanAbsoluteError", "rank_test_MeanAbsoluteError"]
    ]
)

`mean_test_MeanAbsoluteError` is the average forecasting error over the expanding validation windows. Because the scorer is MAE, lower is better. Both lag orders are selected without observing the untouched test period.

In [ ]:
fh = ForecastingHorizon(y_test.index, is_relative=False)
choquet_forecast = search.predict(fh=fh)
choquet_interval = search.predict_interval(fh=fh, coverage=0.95)

classical_ar_forecast = classical_search.predict(fh=fh)

metrics = pd.DataFrame(
    {
        "MAE": [
            MeanAbsoluteError()(y_test, choquet_forecast),
            MeanAbsoluteError()(y_test, classical_ar_forecast),
        ],
        "MAPE": [
            MeanAbsolutePercentageError()(y_test, choquet_forecast),
            MeanAbsolutePercentageError()(y_test, classical_ar_forecast),
        ],
    },
    index=["Choquet autoregression", "classical AR"],
)
display(metrics)
display(choquet_interval.head())

In [ ]:
axis = y_train.iloc[-24:].plot(figsize=(10, 4), label="train")
y_test.plot(ax=axis, label="test")
choquet_forecast.plot(ax=axis, label="Choquet")
classical_ar_forecast.plot(ax=axis, label="classical AR")
axis.legend();

## 4. Interpret the lag capacity

The fitted capacity belongs to the final forecaster inside the selected pipeline. Shapley indices summarize the importance of each lag; pairwise interaction indices indicate complementarity (positive) or redundancy (negative) between lags.

In [ ]:
selected_pipeline = search.best_forecaster_
selected_model = selected_pipeline.steps_[-1][1]
print("AR order:", selected_model.order_)
print("phi:", selected_model.phi_)
print("AIC / BIC:", selected_model.aic(), selected_model.bic())
print("Shapley indices:", shapley_indices(selected_model.capacity_))
print("Pairwise interactions:", pairwise_interactions(selected_model.capacity_))

## 5. Optional exogenous variables and updating

Pass time-indexed `X` to `fit` and future `X` to `predict`. Recursive forecasting needs exogenous rows for every intermediate step. New observations can be incorporated through `update(y_new, X=X_new, update_params=True)`, following the standard `sktime` contract.